In [ ]:
import gdown

# 建立 檔案ID 與 儲存名稱 的對照表
files = {
    '1IKYmauZ3HfFovtzggpfuYRfjfvafQXl7': 'Option_2020.zip',
    '1EmoYj8zv69EfC-lYsu0wITkrmgR7SLXg': '2018~2024收盤價.csv',
    '1xPp8kE2dvIwdle2o468xaVFN1dW6rS2f': '2018~2024臺指選擇權結算.csv'
}

for file_id, output_name in files.items():
    print(f"正在下載: {output_name}")
    gdown.download(id=file_id, output=output_name, quiet=False)

import os

for file_id, output_name in files.items():
    gdown.download(id=file_id, output=output_name, quiet=True)
    if output_name.endswith('.zip'):
        # 解壓到以檔名命名的資料夾
        folder_name = output_name.replace('.zip', '')
        !unzip -o {output_name} -d /content/


正在下載: Option_2020.zip


Downloading...
From (original): https://drive.google.com/uc?id=1IKYmauZ3HfFovtzggpfuYRfjfvafQXl7
From (redirected): https://drive.google.com/uc?id=1IKYmauZ3HfFovtzggpfuYRfjfvafQXl7&confirm=t&uuid=5412a8e1-2373-4882-9845-a3fbaa1f7227
To: /content/Option_2020.zip
100%|██████████| 376M/376M [00:04<00:00, 80.7MB/s]


正在下載: 2018~2024收盤價.csv


Downloading...
From: https://drive.google.com/uc?id=1EmoYj8zv69EfC-lYsu0wITkrmgR7SLXg
To: /content/2018~2024收盤價.csv
100%|██████████| 53.1k/53.1k [00:00<00:00, 21.1MB/s]


正在下載: 2018~2024臺指選擇權結算.csv


Downloading...
From: https://drive.google.com/uc?id=1xPp8kE2dvIwdle2o468xaVFN1dW6rS2f
To: /content/2018~2024臺指選擇權結算.csv
100%|██████████| 9.20k/9.20k [00:00<00:00, 26.6MB/s]


Archive:  Option_2020.zip
  inflating: /content/Option_2020/o20200102.csv  
  inflating: /content/Option_2020/o20200103.csv  
  inflating: /content/Option_2020/o20200106.csv  
  inflating: /content/Option_2020/o20200107.csv  
  inflating: /content/Option_2020/o20200108.csv  
  inflating: /content/Option_2020/o20200109.csv  
  inflating: /content/Option_2020/o20200110.csv  
  inflating: /content/Option_2020/o20200113.csv  
  inflating: /content/Option_2020/o20200114.csv  
  inflating: /content/Option_2020/o20200115.csv  
  inflating: /content/Option_2020/o20200116.csv  
  inflating: /content/Option_2020/o20200117.csv  
  inflating: /content/Option_2020/o20200120.csv  
  inflating: /content/Option_2020/o20200130.csv  
  inflating: /content/Option_2020/o20200131.csv  
  inflating: /content/Option_2020/o20200203.csv  
  inflating: /content/Option_2020/o20200204.csv  
  inflating: /content/Option_2020/o20200205.csv  
  inflating: /content/Option_2020/o20200206.csv  
  inflating: /content/Op

In [ ]:
import csv
from datetime import datetime, date, timedelta

import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import newton


s=[["Date","File","S0","Contract","ContractExpiryDate","Maturity","Rf"]]

ss=[]


def bs_price(S, K, T, r, sigma, option_type='C'):
    """計算 Black-Scholes 理論價格"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'C':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def find_iv(market_price, S, K, T, r, option_type):
    """使用二分法反推隱含波動率"""
    if market_price <= 0 or T <= 0: return np.nan

    # 設定波動率尋找範圍 (0.01% ~ 500%)
    low = 0.01
    high = 4.0
    precision = 1e-3# 容許價格誤差
    max_iter = 100

    # 檢查邊界條件：如果波動率極大時理論價仍低於市價，則無解
    if market_price > bs_price(S, K, T, r, high, option_type):
        return np.nan
    # 如果波動率極小時理論價仍高於市價，則無解
    if market_price < bs_price(S, K, T, r, low, option_type):
        return np.nan

    for i in range(max_iter):
        mid = (low + high) / 2
        price_mid = bs_price(S, K, T, r, mid, option_type)

        if abs(price_mid - market_price) < precision:
            return mid

        if price_mid < market_price:
            low = mid
        else:
            high = mid

    return (low + high) / 2


 # 開啟 CSV 檔案
with open('2018~2024臺指選擇權結算.csv', newline='', encoding='cp950') as csvfile:

    # 讀取 CSV 檔案內容
    rows = csv.reader(csvfile)

    # 以迴圈輸出每一列
    for row in rows:
        if "W" not in row[1] and row[1].isdigit():
            #print(row)
            ss.append(row)
    ss=ss[::-1]
    #print(ss)

# 開啟 CSV 檔案
with open('2018~2024收盤價.csv', newline='', encoding='cp950') as csvfile:

    # 讀取 CSV 檔案內容
    rows = csv.reader(csvfile)

    # 以迴圈輸出每一列
    for row in rows:
        y=['2020']#選擇年分
        for j in y:
            if j in row[0]:
                row.append(row[1])
                row[1]="o"+row[0].split("/")[0]+row[0].split("/")[1]+row[0].split("/")[2]+".csv"
                d1 = date(int(row[0].split("/")[0]), int(row[0].split("/")[1]),int(row[0].split("/")[2]))
                for i in ss:
                    d2=date(int(i[0].split("/")[0]), int(i[0].split("/")[1]), int(i[0].split("/")[2]))
                    delta = d2-d1
                    if delta.days>0:
                        row.append(i[1])
                        row.append(i[0])
                        #print(d1,d2)
                        #print(f"相差天數: {delta.days} 天")
                        row.append(delta.days)
                        row.append(row[2])
                        del row[2]
                        break

                s.append(row)
print(s)
final_summary = [["Date","C_mean", "C_std", "P_mean", "P_std","PCR"]]

# 遍歷剛剛生成的 s 列表 (跳過標題) 進行每日 IV 計算
for daily_info in s[1:]:
    current_date = daily_info[0]
    filename = daily_info[1]
    S0 = float(daily_info[2].replace(',', ''))
    target_contract = str(daily_info[3])
    T_years = float(daily_info[5]) / 365
    Rf = float(daily_info[6])

    call_iv_list = []  # 新增：買權 IV 清單
    put_iv_list = []   # 新增：賣權 IV 清單
    call_volume, put_volume = 0, 0  # 新增：用於計算 PCR

    try:
        with open('Option_2020/'+filename, newline='', encoding='cp950') as csvfile:
            rows = csv.reader(csvfile)
            next(rows) # 跳過第 1 行
            next(rows) # 跳過第 2 行
            for row in rows:
                # 篩選：TXO 商品、正確的到期月份
                if 'TXO' in row[1]:
                    if int(row[7])>=30:
                        mkt_price = eval(row[6])
                        strike = eval(row[2])
                        cp_type = row[4].strip()

                        iv = find_iv(mkt_price, S0, strike, T_years, Rf, cp_type)
                        #print(iv)
                        if not np.isnan(iv):
                            # 根據 CP 類型分類存入
                            if cp_type == 'C':
                                call_volume += 1
                                call_iv_list.append(iv)
                            elif cp_type == 'P':
                                put_volume += 1
                                put_iv_list.append(iv)
    except FileNotFoundError:
        print(f"找不到檔案: {filename}")
        continue

    # 計算統計值 (分別計算 Call / Put)
    res = [current_date]
    for iv_list, name in [(call_iv_list, "Call"), (put_iv_list, "Put")]:
        if iv_list:
            mean_iv = np.mean(iv_list)
            std_iv = np.std(iv_list)
            res.extend([mean_iv, std_iv]) # 依序加入 [C_mean, C_std, P_mean, P_std]
            print(f"日期: {current_date} | {name} Mean IV: {mean_iv:.2%} | Std: {std_iv:.2%}")
        else:
            res.extend([np.nan, np.nan])
    pcr = put_volume / call_volume
    res.append(pcr)
    final_summary.append(res)



with open('Index_411111220.csv', 'w', newline='', encoding='cp950') as f:
    writer = csv.writer(f)
    writer.writerows(s)

# 2. 輸出統計結果檔 (Mean/Std IV)
with open('IV_Statistics.csv', 'w', newline='', encoding='cp950') as f:
    writer = csv.writer(f)
    writer.writerows(final_summary)


[['Date', 'File', 'S0', 'Contract', 'ContractExpiryDate', 'Maturity', 'Rf'], ['2020/01/02', 'o20200102.csv', '12,100.48', '202001', '2020/1/15', 13, '0.0109'], ['2020/01/03', 'o20200103.csv', '12,110.43', '202001', '2020/1/15', 12, '0.0109'], ['2020/01/06', 'o20200106.csv', '11,953.36', '202001', '2020/1/15', 9, '0.0109'], ['2020/01/07', 'o20200107.csv', '11,880.32', '202001', '2020/1/15', 8, '0.0109'], ['2020/01/08', 'o20200108.csv', '11,817.10', '202001', '2020/1/15', 7, '0.0109'], ['2020/01/09', 'o20200109.csv', '11,970.63', '202001', '2020/1/15', 6, '0.0109'], ['2020/01/10', 'o20200110.csv', '12,024.65', '202001', '2020/1/15', 5, '0.0109'], ['2020/01/13', 'o20200113.csv', '12,113.42', '202001', '2020/1/15', 2, '0.0109'], ['2020/01/14', 'o20200114.csv', '12,179.81', '202001', '2020/1/15', 1, '0.0109'], ['2020/01/15', 'o20200115.csv', '12,091.88', '202002', '2020/2/19', 35, '0.0109'], ['2020/01/16', 'o20200116.csv', '12,066.93', '202002', '2020/2/19', 34, '0.0109'], ['2020/01/17', 'o